# REPO ORBIT v0.8.0 — External Git benchmark (CPU / Colab)

Use the `REPO_ORBIT_v0.8.0_EXTERNAL_VALIDATION_R2.zip` downloadable package from this chat. This notebook performs **Linux-only** validation of pinned Requests, Flask and uv histories. It does not establish Windows or macOS acceptance; those must run on native machines. Original engine files are checked by SHA-256 before measurements. No repository source is executed. A completed run needs working network access to GitHub.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io
uploads = files.upload()
assert len(uploads) == 1, 'Upload exactly one R2 ZIP package'
name, blob = next(iter(uploads.items()))
assert name.endswith('.zip'), 'ZIP required'
root = Path('/content/repo_orbit_native_r2')
root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(blob)) as archive:
    for item in archive.infolist():
        target = (root / item.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Unsafe ZIP path')
    archive.extractall(root)
project = root / 'REPO_ORBIT'
assert (project/'validation'/'PINNED_TARGETS.json').is_file(), 'Expected R2 package'
print('Prepared', project)


In [ ]:
import subprocess, sys
cmd = [sys.executable, 'scripts/run_native_matrix.py', '--targets', 'requests', 'flask', 'uv', '--runs', '21', '--commits', '60', '300', '1000', '--verify', 'sample', '--out', 'external-results/colab-linux']
res = subprocess.run(cmd, cwd=project, text=True, capture_output=True)
print(res.stdout)
if res.stderr: print(res.stderr[-2500:])
print('Exit code:', res.returncode, '| 0=all measured, 2=one or more blocked/failed')


In [ ]:
import zipfile, json
out = project/'external-results'/'colab-linux'
audit = project/'external-results'/'colab_audit.json'
cmd = [sys.executable, 'scripts/audit_native_matrix.py', '--inputs', str(out), '--out', str(audit)]
res = subprocess.run(cmd, cwd=project, text=True, capture_output=True)
print(res.stdout)
status = out/'RUN_STATUS.json'
print('Run status:')
print(status.read_text()[:6000] if status.exists() else 'No status file found')
archive_path = Path('/content/REPO_ORBIT_COLAB_LINUX_RESULTS_R2.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    files_to_include = list(out.rglob('*')) if out.exists() else []
    files_to_include += [audit, audit.with_suffix('.md')]
    for f in files_to_include:
        if f.is_file():
            z.write(f, arcname=str(f.relative_to(project)))
    z.write(project/'validation'/'PINNED_TARGETS.json', 'PINNED_TARGETS.json')
files.download(str(archive_path))
print('This is Linux evidence only. A matrix-wide PASS requires native Windows and macOS runs as well.')
